# AI Meeting Intelligence

**Turn a raw meeting recording into timestamped chapters and a searchable record, where every answer points back to the moment it was said.**



---

## 1. Problem and motivation

Meetings are where decisions are made, but most of what is said is forgotten, and finding one answer means replaying hours of recording. Notes are incomplete, and recordings are slow to search.

This project focuses on one practical question:

> How can a recording be turned into something people can quickly understand and search.


## 2. Pipeline

Every model is pre-trained. The work is in connecting them and making the output trustworthy.


| Step | What it does | Tool |
|---|---|---|
| 1. Audio extraction | Video to mono 16 kHz WAV | `ffmpeg` |
| 2. Chunking | Cut at silences (`-30 dB`, 0.3 s), chunks of 4-20 s with 0.2 s padding. If no silence exists in range, cut at the quietest 0.1 s window | `ffmpeg`, NumPy |
| 3. Transcription | Segment-level timestamps, fp16 on GPU (about 3.1 GB of VRAM) | `openai/whisper-large-v3` via `transformers` |
| 4. Cleaning | Unicode normalisation, Arabic diacritics removed, empty or looping chunks replaced with `[unclear speech]` | Python |
| 5. Chapters | Transcript split into windows of at most 3,000 tokens. The LLM returns `start`, `title`, `summary` as JSON. Every `start` is checked against the real transcript | Hosted LLM through OpenRouter, JSON schema |
| 6. Indexing | Chapter text split into 1,000-character pieces (150 overlap), embedded and stored with chapter metadata | `BAAI/bge-m3`, Chroma |
| 7. Question answering | Top-10 pieces are retrieved and passed to the LLM, which must answer only from them, cite `[Source n]`, and give the chapter title and start time. If the answer is not in the text, it says so | Hosted LLM |
| 8. Interface | Chat UI | Streamlit |

## 3. Evaluation and baseline comparison

| Model | Arabic only | English only | Mixed Arabic/English |
|---|---|---|---|
| Whisper large-v3 | Good; errors on names and terms | Very good | Translates or drops speech in language-switching chunks; some repetition loops |
| Qwen3-ASR 1.7B | Readable, more errors | Very good | Often translates Arabic into English |
| Jais-ASR | Best on formal Arabic and dialect samples | Good | Often writes English in Arabic letters |
| Nemotron 3.5 ASR streaming 0.6B | Captured Arabic | Garbled English | Not reliable |

Whisper was chosen as the single model: it handled both languages acceptably, gives timestamps, and uses about 3.1 GB of VRAM.


## 4. Limitations

**Limitations**

- **Mixed-language speech is unreliable.** When the language changes within a few seconds, models translate one language into the other, drop speech, or loop. Pure-language recordings work much better.
- **No speaker labels.** The transcript does not say who is talking, so summaries refer to "a speaker" or "the hosts", and early runs sometimes guessed names wrongly.




In [2]:
# %pip install torch --index-url https://download.pytorch.org/whl/cpu

In [3]:
# pip install chromadb

In [4]:
# !pip install langchain-text-splitters

In [5]:
# pip install -U FlagEmbedding

In [6]:
# !pip install streamlit

In [7]:
# pip install -r requirements.txt

In [8]:
from pathlib import Path
import subprocess

def get_audio(video):
    video = Path(video)
    audio = video.with_name(video.stem + "_audio.wav")

    result = subprocess.run(
        ["ffmpeg", "-y", "-i", str(video), "-vn", "-ac", "1", "-ar", "16000", str(audio)],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg failed:\n{result.stderr[-500:]}")
    if not audio.exists() or audio.stat().st_size == 0:
        raise RuntimeError("Audio file was not created")

    return audio

In [9]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv() 
 
api_key = os.getenv("OPENROUTER") or os.getenv("API_KEY")
if not api_key:
    raise ValueError("Set OPENROUTER or API_KEY in your environment or .env file.")

stt_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
    max_retries=2,
    timeout=120.0
)

In [10]:
import glob, os, shutil, subprocess, json, torch, re,gc
import wave
import numpy as np

# Clean previous chunks
shutil.rmtree("/data/pieces", ignore_errors=True)
os.makedirs("/data/pieces")

# Load audio
video = Path("Searching for The Five.mp4").resolve()

if not video.is_file():
    raise FileNotFoundError(f"Video not found: {video}")

missing = [exe for exe in ("ffmpeg", "ffprobe") if shutil.which(exe) is None]
if missing:
    raise RuntimeError(
        f"Install FFmpeg and add it to PATH; not found: {', '.join(missing)}"
    )

audio = get_audio(video)

# Get audio duration
duration_result = subprocess.run(
    [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(audio)
    ],
    capture_output=True,
    text=True,
    check=True
)
duration = float(duration_result.stdout.strip())


with wave.open(str(audio), "rb") as w:
    sr = w.getframerate()
    samples = np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16).astype(np.float32)

WIN = int(0.1 * sr)
n = len(samples) // WIN
rms = np.sqrt((samples[:n * WIN].reshape(n, WIN) ** 2).mean(axis=1))

def quietest_cut(lo, hi):
    a = int(lo / 0.1)
    b = min(int(hi / 0.1), n)
    if a >= b:
        return lo
    i = a + int(np.argmin(rms[a:b]))
    return (i + 0.5) * 0.1


result = subprocess.run(
    [
        "ffmpeg", "-i", str(audio),
        "-af", "silencedetect=noise=-30dB:d=0.3",
        "-f", "null", "-"
    ],
    capture_output=True,
    text=True
)

log = result.stderr

starts = re.findall(r"silence_start:\s*([\d.]+)", log)
ends = re.findall(r"silence_end:\s*([\d.]+)", log)

silences = [
    (float(s), float(e))
    for s, e in zip(starts, ends)
]


MIN_LEN = 4
TARGET_LEN = 12
MAX_LEN = 20
PADDING = 0.2


boundaries = [0.0]
current = 0.0
fallback_cuts = 0

while duration - current > MAX_LEN:
    candidates = [
        (s + e) / 2
        for s, e in silences
        if current + MIN_LEN <= (s + e) / 2 <= current + MAX_LEN
    ]

    if candidates:

        cut = min(
            candidates,
            key=lambda x: abs(x - (current + TARGET_LEN))
        )
    else:

        cut = quietest_cut(current + MIN_LEN, current + MAX_LEN)
        fallback_cuts += 1

    boundaries.append(cut)
    current = cut

boundaries.append(duration)


chunks = []

for i, (start, end) in enumerate(zip(boundaries[:-1], boundaries[1:])):
    export_start = max(0, start - PADDING) if i > 0 else start
    export_end = min(duration, end + PADDING) if i < len(boundaries) - 2 else end

    output_file = f"/data/pieces/p_{i:04d}.wav"

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-ss", str(export_start),
            "-i", str(audio),
            "-t", str(export_end - export_start),
            "-ac", "1",
            "-ar", "16000",
            output_file
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True
    )

    chunks.append({
        "file": output_file,
        "start": export_start,
        "end": export_end
    })

In [11]:
out = []

for i, chunk in enumerate(chunks):

    with open(chunk["file"], "rb") as audio_file:

        r = stt_client.audio.transcriptions.create(
            model="openai/whisper-large-v3",
            file=audio_file,
            response_format="json"
        )

    out.append({
        "chunk_index": i,
        "start": chunk["start"],
        "end": chunk["end"],
        "text": r.text.strip()
    })

    # Save progress after every chunk
    transcript_dir = Path("data/transcripts")
    transcript_dir.mkdir(parents=True, exist_ok=True)

    with open(
        transcript_dir / "transcript_mix_2.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

    print(
        f"[{chunk['start']:.2f}s - {chunk['end']:.2f}s] "
        f"{r.text}"
    )

print("Transcription completed!")

[0.00s - 16.35s]  Thanks for watching!  Friday.
[15.95s - 35.65s]  Joseph Shones was scouting some back roads in the Plumas National Forest. He spent the better part of the day navigating his Volkswagen in hopes of returning with his family the next day.
[35.25s - 55.35s]  However, due to harsh weather conditions, his venture led to his car becoming stuck around 5:30 p.m.  Miles away from help and with dwindling options, he hopped out of his car to give it a push.  In a strikingly unlucky turn of events though, this physical exertion would lead to him enduring a heart attack at the worst possible place.
[54.95s - 66.25s]  and the worst possible time.  He returns to the confines of his vehicle, enduring the pain and contemplating on what  to do.
[65.85s - 80.45s] A short while later though, he'd catch his biggest break. Two sets of headlights coming from the opposite direction and the shadows of what looked like a woman, a baby, and a group of guys walking towards him while engaging in 

In [12]:
import re
import unicodedata


def clean_text(text):
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"[\u064B-\u065F\u0670\u0640]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def is_bad(text):
    words = text.split()
    if not words:
        return True
    if re.search(r"(.)\1{9,}", text):
        return True
    top = max(set(words), key=words.count)
    return len(words) >= 8 and words.count(top) / len(words) > 0.5


def clean_transcript(chunks):
    out = []
    for c in chunks:
        text = clean_text(c["text"])
        bad = is_bad(text)
        out.append({**c,
                    "text_clean": "[unclear speech]" if bad else text,
                    "masked": bad})
    return out

In [13]:
import json
import time
from openai import OpenAI
from tenacity import retry, stop_after_attempt, wait_random_exponential, retry_if_exception_type
import openai

PROMPT = """You are helping to organize a meeting transcript.

Below is part of a transcript. Each line starts with the time it begins, like [12:30].
Divide this part into 2 to 5 chapters, in time order.

For each chapter give:
- start: the time of the first line of the chapter, copied exactly from the transcript (for example "12:30")
- title: at most eight words
- summary: one sentence about what happens in the chapter

Rules:
- Use only what is written in the transcript. Never invent names, numbers, or events.
- Names may be spelled in different ways. If you are not sure who is speaking, say "a speaker".
- Lines marked [unclear speech] are gaps. Do not treat them as content.
- Write the titles and summaries in English.
-Never say who stated, guessed, or received something unless that person’s name appears in the same line. Otherwise write ‘a speaker’.
-Do not state scores, totals, or results unless they are written in the text.
-Each topic gets one chapter. The start time is the first line about that topic.
-Never say which person stated, guessed, or confirmed something. Write ‘a host’ or ‘the hosts’. You may use a name only for someone being talked about, not for who is speaking.
-Do not add any name, title, or fact that is not written in the text, even if you know it.
-Each chapter’s start time must be the first line about that chapter’s title.

TRANSCRIPT:
"""

schema = {
  "type": "object",
  "properties": {"chapters": {"type": "array", "items": {
      "type": "object",
      "properties": {"start": {"type": "string"},
                     "title": {"type": "string"},
                     "summary": {"type": "string"}},
      "required": ["start", "title", "summary"]}}},
  "required": ["chapters"]}

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
    max_retries=0
)

@retry(
    wait=wait_random_exponential(min=1, max=60),
    stop=stop_after_attempt(5),
    retry=retry_if_exception_type(openai.RateLimitError),
    reraise=True
)
def generate_chapters(window_text):

    return client.chat.completions.create(
        model="nvidia/nemotron-3-ultra-550b-a55b:free",

        extra_body={
            "reasoning": {
                "effort": "minimal"
            }
        },

        messages=[
            {
                "role": "user",
                "content": PROMPT + window_text
            }
        ],

        temperature=0.2,
        max_tokens=10000,

        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "chapters",
                "strict": True,
                "schema": schema
            }
        }
    )

In [28]:
def generate_answer(query, results):
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=api_key,
        max_retries=0
    )

    context = ""

    for i, (doc, metadata) in enumerate(zip(results["documents"][0], results["metadatas"][0]), 1):
        context += f"""
[Source {i}]
Chapter: {metadata["chapter"]}
Chapter start: {metadata["chapter_start"]}
Transcript:
{doc}

"""

    response = client.chat.completions.create(
        model="nvidia/nemotron-3-ultra-550b-a55b:free",

        messages=[
            {
                "role": "system",
                "content": """
You are an assistant answering questions about a recording.

Instructions:
- Answer using only the provided transcript passages.
- Do not invent information or use outside knowledge.
- Distinguish reported facts from opinions or interpretations.
- If the transcript does not contain enough information, say so.
- Include citations using [Source 1], [Source 2], etc.
- Include the chapter title and chapter start time.
"""
            },
            {
                "role": "user",
                "content": f"""
Retrieved transcript:

{context}

Question:
{query}

Answer:
"""
            }
        ],

        temperature=0.2,
        max_tokens=1500
    )

    return response.choices[0].message.content

In [15]:
from transformers import AutoTokenizer
import json

with open(transcript_dir / "transcript_mix_2.json", encoding="utf-8") as f:
    raw = json.load(f)
clean = clean_transcript(raw)
with open(transcript_dir / "transcript_clean.json", "w", encoding="utf-8") as f:
    json.dump(clean, f, ensure_ascii=False, indent=2)

def mmss(sec):
    return f"{int(sec // 60):02d}:{int(sec % 60):02d}"

lines = [f"[{mmss(c['start'])}] {c['text_clean']}" for c in clean]
with open(transcript_dir / "transcript.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base")
def n_tokens(text):
    return len(tokenizer.encode(text, add_special_tokens=False))

def make_windows(chunks, max_tokens=3000):
    windows, current, count = [], [], 0
    for c in chunks:
        line = f"[{mmss(c['start'])}] {c['text_clean']}"
        t = n_tokens(line)
        if current and count + t > max_tokens:
            windows.append("\n".join(current))
            current, count = [], 0
        current.append(line)
        count += t
    if current:
        windows.append("\n".join(current))
    return windows

windows = make_windows(clean)

all_chapters = []

for i, window_text in enumerate(windows):

    print(f"\nProcessing window {i+1}/{len(windows)}")

    try:
        response = generate_chapters(window_text)

        choice = response.choices[0]

        if choice.finish_reason == "length":
            usage = (
                response.usage.model_dump()
                if response.usage else {}
            )

            print("Model:", response.model)
            print("Finish reason:", choice.finish_reason)
            print("Usage:", usage)

            details = usage.get("completion_tokens_details") or {}

            print(
                "Reasoning tokens:",
                details.get("reasoning_tokens")
            )

            print("Returned content:", repr(choice.message.content))

            raise ValueError("Response was truncated.")

        content = choice.message.content

        if not content:
            raise ValueError("Empty model response.")

        chapters = json.loads(content)["chapters"]

        # Validate timestamps
        valid_starts = {
            line.split("]")[0][1:]
            for line in window_text.splitlines()
            if line.startswith("[") and "]" in line
        }

        for ch in chapters:
            if ch["start"] not in valid_starts:
                raise ValueError(
                    f"Invalid timestamp: {ch['start']}"
                )

        all_chapters.extend(chapters)

        # Save progress after every window
        with open(
            transcript_dir / "chapters_progress.json",
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                all_chapters,
                f,
                ensure_ascii=False,
                indent=2
            )

        print("Model:", response.model)
        print("Chapters generated:", len(chapters))

        time.sleep(2)

    except Exception as e:
        print(f"Window {i+1} failed: {e}")
        break
for ch in all_chapters:
    print(
        ch["start"], "|",
        ch["title"], "|",
        ch["summary"]
    )

with open(
    transcript_dir / "final_chapters.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {"chapters": all_chapters},
        f,
        ensure_ascii=False,
        indent=2
    )

print("Number of chunks:", len(clean))
print("Number of windows:", len(windows))

for i, window in enumerate(windows):
    count = sum(n_tokens(line) for line in window.splitlines())
    print(f"Window {i+1}: {count} tokens")

c:\Users\ps4so\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Processing window 1/2
Model: nvidia/nemotron-3-ultra-550b-a55b:free
Chapters generated: 5

Processing window 2/2
Model: nvidia/nemotron-3-ultra-550b-a55b:free
Chapters generated: 5
00:00 | Joseph Shones' Mountain Ordeal | Joseph Shones suffers a heart attack while his car is stuck in snow, witnesses mysterious figures with headlights and flashlights who ignore his pleas, and later walks eight miles to a lodge passing an abandoned Mercury Montego.
03:08 | Five Friends Attend Basketball Game | Five men from the Gateway Gators team, known as The Boys, drive to a Chico State basketball game, buy snacks at a convenience store, and begin their return trip home.
07:09 | Wrong Turn Leads to Disappearance | The group takes an unexplained detour east onto Oroville-Quincy Highway, drives until the road ends, continues on a dirt trail until their car becomes stuck in a snow drift, and vanishes into the night.
09:18 | Abandoned Car Found, Search Halted | A forest ranger discovers the unlocked Merc

In [16]:
# from llama_cpp import Llama

# llm = Llama.from_pretrained(
#     repo_id="Qwen/Qwen3-8B-GGUF",
#     filename="Qwen3-8B-Q4_K_M.gguf",
#     n_gpu_layers=-1,
#     n_ctx=8192
# )

In [17]:
# PROMPT = """You are helping to organize a meeting transcript.

# Below is part of a transcript. Each line starts with the time it begins, like [12:30].
# Divide this part into 2 to 5 chapters, in time order.

# For each chapter give:
# - start: the time of the first line of the chapter, copied exactly from the transcript (for example "12:30")
# - title: at most eight words
# - summary: one sentence about what happens in the chapter

# Rules:
# - Use only what is written in the transcript. Never invent names, numbers, or events.
# - Names may be spelled in different ways. If you are not sure who is speaking, say "a speaker".
# - Lines marked [unclear speech] are gaps. Do not treat them as content.
# - Write the titles and summaries in English.

# TRANSCRIPT:
# """



# schema = {
#   "type": "object",
#   "properties": {"chapters": {"type": "array", "items": {
#       "type": "object",
#       "properties": {"start": {"type": "string"},
#                      "title": {"type": "string"},
#                      "summary": {"type": "string"}},
#       "required": ["start", "title", "summary"]}}},
#   "required": ["chapters"]}

# response = llm.create_chat_completion(
#     messages=[{"role": "user", "content": PROMPT + "\n\n" + window_text + "\n\n/no_think"}],
#     response_format={"type": "json_object", "schema": schema},
#     temperature=0.2,
#     max_tokens=800,
# )

In [18]:
# pip install langchain

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [20]:
import json


with open(transcript_dir / "final_chapters.json", "r", encoding="utf-8") as f:
    final_chapters = json.load(f)
with open(transcript_dir / "transcript_clean.json", "r", encoding="utf-8") as f:
    transcript_clean = json.load(f)

def to_seconds(timestamp):
    minutes, seconds = map(int, timestamp.split(":"))
    return minutes * 60 + seconds


for i, chapter in enumerate(final_chapters["chapters"]):
    chapter_start = to_seconds(chapter["start"])

    if i + 1 < len(final_chapters["chapters"]):
        chapter_end = to_seconds(final_chapters["chapters"][i + 1]["start"])
    else:
        chapter_end = float("inf")

    for j in transcript_clean:
        if chapter_start <= j["start"] < chapter_end:
            j["chapter"] = chapter["title"]
            j["chapter_summary"] = chapter["summary"]
            j["chapter_index"] = i
            j["chapter_start"] = chapter["start"]

with open(transcript_dir / "transcript_clean.json", "w", encoding="utf-8") as f:
    json.dump(transcript_clean, f, ensure_ascii=False, indent=2)

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chapters_text = {}

for chunk in transcript_clean:
    index = chunk["chapter_index"]

    if index not in chapters_text:
        chapters_text[index] = {
            "chapter": chunk["chapter"],
            "chapter_summary": chunk["chapter_summary"],
            "start": chunk["chapter_start"],
            "text": ""
        }

    chapters_text[index]["text"] += chunk["text_clean"] + " "


rag_chunks = []

for index, chapter in chapters_text.items():
    pieces = splitter.split_text(chapter["text"])

    for piece in pieces:
        rag_chunks.append({
            "text": piece,
            "chapter": chapter["chapter"],
            "chapter_summary": chapter["chapter_summary"],
            "chapter_start": chapter["start"],
            "chapter_index": index
        })

with open(transcript_dir/ "rag_chunks.json", "w", encoding="utf-8") as f:
    json.dump(rag_chunks, f, ensure_ascii=False, indent=2)

print("Number of RAG chunks:", len(rag_chunks))
print(rag_chunks[0])


Number of RAG chunks: 28
{'text': "Thanks for watching! Friday. Joseph Shones was scouting some back roads in the Plumas National Forest. He spent the better part of the day navigating his Volkswagen in hopes of returning with his family the next day. However, due to harsh weather conditions, his venture led to his car becoming stuck around 5:30 p.m. Miles away from help and with dwindling options, he hopped out of his car to give it a push. In a strikingly unlucky turn of events though, this physical exertion would lead to him enduring a heart attack at the worst possible place. and the worst possible time. He returns to the confines of his vehicle, enduring the pain and contemplating on what to do. A short while later though, he'd catch his biggest break. Two sets of headlights coming from the opposite direction and the shadows of what looked like a woman, a baby, and a group of guys walking towards him while engaging in conversation. After realizing that this could be his ticket hom

In [22]:
from FlagEmbedding import BGEM3FlagModel
import json

model = BGEM3FlagModel(
    'BAAI/bge-m3',
    use_fp16=True
)

texts = [chunk["text"] for chunk in rag_chunks]

embeddings = model.encode(
    texts,
    batch_size=8,
    max_length=8192
)

for i, chunk in enumerate(rag_chunks):
    # Convert NumPy array to a standard list so it can be serialized to JSON
    chunk["embedding"] = embeddings["dense_vecs"][i].tolist()

with open(transcript_dir / "rag_chunks.json", "w", encoding="utf-8") as f:
    json.dump(rag_chunks, f, ensure_ascii=False, indent=2)

Fetching 30 files:  67%|██████▋   | 20/30 [00:00<00:00, 195.16it/s]c:\Users\ps4so\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ps4so\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Fetching 30 files:  93%|█████████▎| 28/30 [00:20<00:00, 

In [23]:
import chromadb, json

with open(transcript_dir / "rag_chunks.json", "r", encoding="utf-8") as f:
    rag_chunks = json.load(f)
client = chromadb.PersistentClient(path="/data/chromadb")
collection = client.get_or_create_collection(
    name="distractible_transcript"
)

documents = [chunk["text"] for chunk in rag_chunks]

embeddings = [chunk["embedding"] for chunk in rag_chunks]

metadatas = [
    {
        "chapter": chunk["chapter"],
        "chapter_summary": chunk["chapter_summary"],
        "chapter_start": chunk["chapter_start"],
        "chapter_index": chunk["chapter_index"]
    }
    for chunk in rag_chunks
]

ids = [f"chunk_{i}" for i in range(len(rag_chunks))]

collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas
)

In [24]:
print(collection.count())

28


In [25]:
def ask_rag(query, n_results=10):


    query_embedding = model.encode(
        [query],
        batch_size=1
    )["dense_vecs"][0].tolist()


    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )


    if not results["documents"] or not results["documents"][0]:
        return "I couldn't find relevant information in the transcript."


    answer = generate_answer(query, results)


    return answer

In [29]:
# What happened during the GoPro visit, what issue did the speaker have with the equipment, and what were the new camera's capabilities?
print(ask_rag("What happened during the GoPro visit, what issue did the speaker have with the equipment, and what were the new camera's capabilities?"))

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 187.83it/s]


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  1.27it/s]


The provided transcript passages do not contain any information about a GoPro visit, equipment issues with a camera, or the capabilities of a new camera. The transcripts exclusively discuss the disappearance of five men after a basketball game, Joseph Shones' testimony about his mountain ordeal, the discovery of an abandoned Mercury Montego, and the later discovery of a body at a forest service trailer. There is no mention of GoPro or any camera equipment in the sources provided.
